# 6 · A person in the loop

A node that needs a person raises `Ask`; its return type says what the answer is and how to ask. Only what reads it waits, and the run ends `waiting` with every question on it. `run.answer(unit, value)` writes an answer in and `run.resume()` goes on.

In [ ]:
from typing import Annotated

from conductor import Ask, GraphRecord, NodeRecord, NodeRegistry, Param, Series
from conductor.widgets import Textarea

registry = NodeRegistry()


@registry.node("text-uppercase", category="text")
def uppercase(text: str) -> str:
    return text.upper()


@registry.node("text-split", category="text")
def split(text: str, separator: str = ",") -> Series[str]:
    return text.split(separator)


@registry.node("text-join", category="text")
def join(parts: Series[str], separator: str = "") -> str:
    return separator.join(parts)


Decision = Annotated[str, Param(title="Decision")]


@registry.node("approve", title="Approve", description="Asks a person to approve the proposal", category="review")
def approve(proposal: Annotated[str, Param(title="Proposal", widget=Textarea())] = "") -> Decision:
    """Asks a person to approve a proposal; the person's answer is the node's output."""
    raise Ask("Approve or rewrite the proposal.", default=proposal, widget=Textarea())

## One question

`shout` waits for `approve`; `other` runs in the same leg.

In [ ]:
graph = registry.load(GraphRecord(
    nodes=[
        NodeRecord(id="approve", node="approve", values={"proposal": "Ship it"}),
        NodeRecord(id="shout", node="text-uppercase"),
        NodeRecord(id="other", node="text-uppercase", values={"text": "independent"}),
    ],
    edges=[("approve.result", "shout.text")],
))

run = graph.run()

print(run.status)
for unit in run.waiting:
    print(unit.id, unit.row, unit.prompt)
    print(unit.questions)
print("ran in this leg:", sorted(node_id for node_id, outputs in run.results.items() if outputs["result"] is not None))

The next leg:

In [ ]:
from conductor.events import RunEnd

answered = run.answer(run.waiting[0], "Ship it on Monday")

events = [event async for event in answered.stream()]
started = [event.node_id for event in events if event.type == "node_start"]
ended = events[-1]  # the last event is the run's end, which carries the run
assert isinstance(ended, RunEnd)

print(ended.run.status)
print(ended.run.results["shout"]["result"])
print("started again:", started)

## A question per row

A node on rows asks once per row. Each waiting unit has its own id, the node and its row (`approve@1`), and an answer is given by it; `run.answer({id: value, ...})` gives several at once, refusing every wrong one together. A unit left out goes on waiting.

In [ ]:
graph = registry.load(GraphRecord(
    nodes=[
        NodeRecord(id="drafts", node="text-split", values={"text": "first draft,second draft"}),
        NodeRecord(id="approve", node="approve"),
        NodeRecord(id="joined", node="text-join", values={"separator": " | "}),
    ],
    edges=[("drafts.result", "approve.proposal"), ("approve.result", "joined.parts")],
))

run = graph.run()
print(run.status, [(unit.row, unit.questions[0].get("default")) for unit in run.waiting])

run = run.answer({unit.id: f"{unit.questions[0].get('default')}, approved" for unit in run.waiting})

done = run.resume()
print(done.status, done.results["joined"]["result"])

Between legs a run is JSON, so it can wait days in a host's table:

```python
saved = run.to_json()
run = graph.load_run(saved)        # in another process, against the same graph
```

## How the return type asks

`-> bool` is a yes or no, a `Literal` one of its values, `str`, `int`, `float` or `date` a value, `Table[R]` records, and `Series[X]` several of `X`. A frozen dataclass is a form, one question per field: a field's own default is proposed, and `defaults=` and `widgets=` on `Ask` go by field name. Each question is the field's record as `describe()` writes it, plus what the `Ask` added, so a host draws it as it draws an input.

In [ ]:
import datetime
from dataclasses import dataclass
from typing import Literal

from conductor import Choice


@dataclass(frozen=True)
class Review:
    approved: bool
    note: str | None
    due: datetime.date = datetime.date(2026, 12, 1)


@registry.node("review")
def review(proposal: str) -> Review:
    raise Ask(f"Review: {proposal}", widgets={"note": Textarea(rows=3)})


@registry.node("labels")
def labels(proposal: str) -> Series[Literal["urgent", "legal", "money"]]:
    raise Ask("Which labels?", default=["urgent"])


@registry.node("owner")
def owner(proposal: str) -> str:
    raise Ask("Who owns it?", choices=[Choice("ida", "Ida B."), Choice("bo", "Bo K.")])


graph = registry.load(GraphRecord(
    nodes=[
        NodeRecord(id="r", node="review", values={"proposal": "Ship it"}),
        NodeRecord(id="l", node="labels", values={"proposal": "Ship it"}),
        NodeRecord(id="o", node="owner", values={"proposal": "Ship it"}),
    ],
))
run = graph.run()
for unit in run.waiting:
    print(unit.id, unit.prompt, [(q["name"], q["required"], q.get("default"), q.get("choices")) for q in unit.questions])

A form is answered by field name, leaving out what has a default or takes a blank; a pick of several is a list of the choices. Every wrong answer is refused at once:

In [ ]:
from conductor import InvalidCallError

try:
    run.answer({"r": {"note": "fine"}, "l": ["urgent", "urgent"], "o": "carl"})
except InvalidCallError as wrong:
    print([(problem.loc, problem.ctx.get("question"), problem.ctx["reason"]) for problem in wrong.errors()])

done = run.answer({"r": {"approved": True}, "l": ["urgent", "money"], "o": "ida"}).resume()
print(done.status, done.outputs)